In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import gc
import os
import sys
import functools
os.environ["HF_HOME"] = "/cs/student/project_msc/2025/csml/katayama/huggingface_cache_marin/jellyfish"
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.std import tqdm
import itertools
import glob
import re
import numpy as np
from sklearn.decomposition import IncrementalPCA
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

In [2]:
# Import custom module
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

%load_ext autoreload
%autoreload 2

from modules.experiment_helpers import set_seed

In [3]:
model_id = "marin-community/marin-8b-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
subject = "all"
batch_size = 2
Trained = False
Parameter_Obtained = True
set_seed(42)

cuda


In [4]:
if not Trained:
    print(f"loading {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision="jellyfish")
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.bfloat16,
        device_map="auto", 
        quantization_config=quantization_config,
        revision="jellyfish"
    )
    model.eval()

loading marin-community/marin-8b-base...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/335 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`rope_scaling`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=4096


... (4 outputs left out)

model-00001-of-00004.safetensors:   0%|          | 0.00/9.83G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/9.87G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/8.22G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
if not Trained: 
    print(f"loading MMLU ({subject})...")
    dataset = load_dataset("cais/mmlu", subject, split="test")

loading MMLU (all)...


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

all/auxiliary_train-00000-of-00001.parqu(…):   0%|          | 0.00/47.5M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

In [6]:
if not Trained: 
    choices = ["A", "B", "C", "D"]
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]
    prompt_template = "Question: {question}\nA. {a}\nB. {b}\nC. {c}\nD. {d}\nAnswer: "

In [7]:
if not Trained:
    prompts = []
    answers = []
    for sample in dataset:
        p = prompt_template.format(
            question=sample['question'],
            a=sample['choices'][0],
            b=sample['choices'][1],
            c=sample['choices'][2],
            d=sample['choices'][3]
        )
        prompts.append(p)
        answers.append(sample['answer'])

    correct_count = 0
    total_count = len(dataset)

In [8]:
if not Trained:
    # Left padding for batch processing
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Get token IDs for the choices
    choices = ["A", "B", "C", "D"]
    choice_ids = [tokenizer.encode(c, add_special_tokens=False)[-1] for c in choices]

    correct_count = 0
    total_count = len(dataset)

    print("="*50)
    print(f"Starting MMLU Accuracy Evaluation (Logit-based)...")
    print(f"Total questions: {total_count}")
    print("="*50)

    for i in range(0, total_count, batch_size):
        batch_prompts = prompts[i:i+batch_size]
        batch_answers = answers[i:i+batch_size]
        
        if not batch_prompts: continue

        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        try:
            # --------------------------------------------------
            # STEP 1: Get logits via forward pass
            # --------------------------------------------------
            with torch.inference_mode():
                outputs = model(**inputs)
                
            # --------------------------------------------------
            # STEP 2: Accuracy evaluation based on logits
            # --------------------------------------------------
            next_token_logits = outputs.logits[:, -1, :]
            
            # Extract the logits corresponding to A, B, C, and D from the entire vocabulary
            choice_logits = next_token_logits[:, choice_ids]
            
            # Get the index (0, 1, 2, 3) of the highest logit among the four choices
            predictions = torch.argmax(choice_logits, dim=-1)
            
            # Compare with the ground truth answers
            for j in range(len(batch_prompts)):
                if predictions[j].item() == batch_answers[j]:
                    correct_count += 1
                    
        except torch.cuda.OutOfMemoryError:
            print(f"\nOOM Error at batch {i}. Skipping...")
            pass
        finally:
            torch.cuda.empty_cache()
            gc.collect()

    # ==========================================
    # Final Accuracy Report
    # ==========================================
    if total_count > 0:
        accuracy = correct_count / total_count
        print("\n" + "="*50)
        print(f"Subject: {subject}")
        print(f"Total Questions: {total_count}")
        print(f"Correct Answers: {correct_count}")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print("="*50)

Starting MMLU Accuracy Evaluation (Logit-based)...
Total questions: 14042



Subject: all
Total Questions: 14042
Correct Answers: 8360
Accuracy: 59.54%
